# Version 22 — Economic MPC (direct reward-weighted optimization)

**Hypothesis:** selecting AHU modes and VAV profiles by a forecast of the competition's **actual 0.6/0.2/0.2 normalized reward** outperforms V12's hand-tuned comfort/energy surrogate.

- **Objective:** at 5-, 20-, and 50-minute forecast checkpoints, minimize 0.6 × the official 5-K-knee comfort regret, plus 0.2 × normalized electricity/gas cost, plus 0.2 × normalized electricity/gas carbon.
- **Comfort:** uses the official linear-then-quadratic regret. Per-zone occupancy is not observed: equal occupied-zone weighting is used, and global occupancy only suppresses comfort regret if metadata establishes the observation encoding and the observed total is zero.
- **Economics:** 24-hour weekday/weekend electricity tariffs and hourly electricity carbon intensity from the official starter kit, gas average-price fallback (month is not observed), and public normalization scales 80 kW electricity / 205 kW gas per AHU. HVAC electrical and thermal consumption are **physics-inspired forecasts, not measured energy**. Recirculation used for forecasting is the same fraction issued to the AHU.
- **Dynamics:** bounded RLS learns five-minute action-conditioned thermal response per zone. Six discrete operating points and at most three VAV profiles per zone keep inference bounded; unsafe off/one-sided modes are guarded during severe violations. The optimizer uses receding horizon, not a complete plant simulator.
- **Ablation:** set `USE_ECONOMIC_OBJECTIVE = False` to restore V12's heuristic decision score on the **same** expanded candidate action grid. Retain V12 unchanged as a separate baseline.
- **Submission:** run this notebook in Kaggle with internet disabled. The final cell invokes `SmartBuildingsInferenceServer(TransferPolicy).run()`; the server creates `submission.parquet`. Only standard NumPy and the competition gateway are required.

**Important limitations:** the official reward cannot be computed exactly from observation fields because measured electric/gas draw and per-zone occupancy are not exposed; the planning objective is an **approximation of the official reward**, with exact published weighting and normalization structure. No official simulator or leaderboard improvement has been measured. Validate V12 vs V22 on identical complete 8,640-step scenarios before accepting the hypothesis.

Source of reward: `smart_buildings/smart_control/reward/setpoint_energy_carbon_regret.py` in the [official starter kit](https://github.com/smart-buildings-2026/starter-kit).


In [ ]:
"""Version 22: economic MPC with an approximate official-reward objective.

The *weights and normalization* match the organizer's regret reward:
0.6 occupancy-weighted comfort, 0.2 normalized electricity + gas cost,
0.2 normalized electricity + gas carbon. Power is estimated from
commanded airflow, fan pressure, mixing and reheat, NOT measured; zone
occupancy is unavailable, so the comfort allocation is approximate.

Keeps bounded online per-zone RLS and a small receding-horizon search.
USE_ECONOMIC_OBJECTIVE=False is a controlled V12-surrogate ablation.
No full simulator reward or performance improvement is claimed.
"""
import re
from typing import NamedTuple

import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class OperatingPoint(NamedTuple):
    """One physical AHU/plant operating point in Kelvin and pascals."""

    mode: str
    running: bool
    supply_k: float
    fan_pa: float
    water_k: float
    water_on: bool


class ZoneResponseModel:
    """Fit one zone's five-minute Kelvin change using bounded online RLS.

    Regression: ΔT = b0 * ((Tout-T)/20) + b1 * cooling + b2 * reheat + b3.
    The features encode *issued actions*, so the model can compare the
    consequences of candidate commands, unlike a passive trend estimator.
    Coefficient signs are constrained by physical direction; the priors are
    used from step one and the covariance limits the size of updates.
    """

    MAX_COVARIANCE_DIAGONAL = 8.0
    INITIAL_COVARIANCE = (0.45, 0.55, 0.55, 0.12)

    def __init__(self):
        self.weights = np.array([0.035, -0.16, 0.20, 0.0], dtype=np.float64)
        self.covariance = np.diag(self.INITIAL_COVARIANCE)
        self.last_temperature = None
        self.last_features = None
        self.samples = 0

    @staticmethod
    def features(temp, outdoor, run, supply, damper, valve, water_on,
                 water_temp, recirculation=0.70):
        """Produce action-dependent ambient/cooling/reheat/bias features.

        Temperatures use Kelvin; damper/valve are fractions, not commands.
        """
        outdoor = outdoor if np.isfinite(outdoor) else temp
        ambient = np.clip((outdoor - temp) / 20.0, -3.0, 3.0)
        # AHU cooling coil cannot heat. Natural outdoor mixing can cool air
        # further than the supply setpoint even with the chiller inactive.
        mixed_air = recirculation * temp + (1.0 - recirculation) * outdoor
        delivered = min(supply, mixed_air)
        cooling = run * damper * np.clip((temp - delivered) / 12.0, 0.0, 2.0)
        reheat = (run * damper * valve * water_on *
                  np.clip((water_temp - temp) / 55.0, 0.0, 2.0))
        return np.array([ambient, cooling, reheat, 1.0], dtype=np.float64)

    def observe(self, current_temperature):
        """Update RLS coefficients from the previous issued action's outcome.

        Forgetting factor 0.995; clip residual and physically directional
        coefficients; discard implausible >1.5K five-minute jumps.
        """
        if self.last_temperature is None or self.last_features is None:
            return
        delta = current_temperature - self.last_temperature
        if not np.isfinite(delta) or abs(delta) > 1.5:
            return  # sensor discontinuity, not ordinary thermal response
        x = self.last_features
        px = self.covariance @ x
        denominator = 0.995 + x @ px
        if not np.isfinite(denominator) or denominator <= 1e-10:
            # Recover from ill-conditioned identification rather than letting
            # one numerical failure contaminate every later action.
            self.covariance = np.diag(self.INITIAL_COVARIANCE)
            return

        gain = px / denominator
        predicted = float(self.weights @ x)
        self.weights += gain * np.clip(delta - predicted, -0.8, 0.8)
        self.weights = np.clip(
            self.weights,
            [0.002, -0.65, 0.004, -0.12],
            [0.20, -0.004, 0.70, 0.12],
        )

        self.covariance = (self.covariance - np.outer(gain, px)) / 0.995
        self.covariance = 0.5 * (self.covariance + self.covariance.T)

        # RLS forgetting otherwise inflates unexcited covariance directions
        # by 1/0.995 at every step; this episode lasts 8,640 steps.
        diagonal = np.diag(self.covariance)
        if (not np.all(np.isfinite(self.covariance)) or
                np.any(diagonal <= 0.0)):
            self.covariance = np.diag(self.INITIAL_COVARIANCE)
        else:
            largest = float(np.max(diagonal))
            if largest > self.MAX_COVARIANCE_DIAGONAL:
                self.covariance *= self.MAX_COVARIANCE_DIAGONAL / largest
        self.samples += 1

    def predict_delta(self, features):
        return float(np.clip(self.weights @ features, -0.50, 0.50))

    def remember(self, temperature, features):
        self.last_temperature = float(temperature)
        self.last_features = features.copy()

    def clear_pending(self):
        self.last_temperature = None
        self.last_features = None


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Select an approximately economical action using online learned dynamics.

    At every step: learn the preceding action response, compare a bounded
    grid of AHU modes and per-zone VAV profiles, and minimize predicted
    *negative official reward* over the next 5/20/50 minutes. Physical energy
    draw remains an estimated plant model, not an exact simulator forecast.
    """

    # Receding-horizon check points, in five-minute control steps.
    # Limit to six AHU modes x three VAV profiles; no nested simulator calls.
    LEARN_DYNAMICS = True
    USE_ECONOMIC_OBJECTIVE = True
    OPTIMIZE_ENERGY = True  # V12 baseline-ablation switch only
    PLAN_STEPS = (1, 4, 10)
    DISCOMFORT_WEIGHT = 1.0
    ENERGY_WEIGHT = 1.0

    # Organizer's actual score is a normalized weighted regret.
    REWARD_COMFORT = 0.6
    REWARD_COST = 0.2
    REWARD_CARBON = 0.2
    ELECTRICITY_NOMINAL_KW_PER_AHU = 80.0
    GAS_NOMINAL_KW_PER_AHU = 205.0
    GAS_USD_PER_KWH = 8.0 / 293.07107  # annual-average fallback; month unavailable
    GAS_KG_CO2_PER_KWH = 53.12 / 293.07107
    ELECTRICITY_CO2_KG_PER_KWH = (
        0.08819666493, 0.08779190866, 0.08787607686, 0.08783054163,
        0.08800279618, 0.08819648183, 0.08970663283, 0.09397947901,
        0.09885868291, 0.10078535210, 0.10138668660, 0.10177956120,
        0.10259191680, 0.10344037360, 0.10413802940, 0.10473592920,
        0.10207144660, 0.09704226176, 0.09357895651, 0.09246355045,
        0.09172914657, 0.09069209747, 0.08976552213, 0.08899950995,
    )
    # Source: official starter-kit electricity_energy_cost.py. No month or
    # exact per-zone occupancy is exposed by the transfer policy observations.
    OPERATING_POINTS = (
        OperatingPoint('off', False, 305., 200., 310., False),
        OperatingPoint('vent', True, 305., 460., 310., False),
        OperatingPoint('mild_cool', True, 295., 570., 310., False),
        OperatingPoint('cool', True, 289., 760., 310., False),
        OperatingPoint('heat', True, 305., 760., 348., True),
        OperatingPoint('mixed', True, 291., 800., 340., True),
    )

    def __init__(self, building_meta_data, action_spec):
        """Map variable-sized building sensors and actuator locations from metadata."""
        super().__init__(building_meta_data, action_spec)
        m = building_meta_data
        self.continuous_count = int(m['n_continuous_action'])
        self.discrete_count = int(m['n_discrete_action'])
        self.fields = {name: i for i, name in enumerate(m['observation_field_names'])}
        self.ahu_supply_actions = [int(x) for x in m['temp_indices']]
        self.ahu_pressure_actions = [int(x) for x in m['pressure_indices']]
        self.ahu_recirc_actions = [int(x) for x in m['recirculation_indices']]
        self.water_actions = [int(x) for x in m['water_indices']]
        self.water_temp_action = self.water_actions[0] if self.water_actions else None
        n_ahu = len(self.ahu_supply_actions)
        self.water_pressure_action = (self.ahu_pressure_actions[n_ahu]
                                      if len(self.ahu_pressure_actions) > n_ahu else None)
        self.water_run_action = self.discrete_count - 1
        self.comfort = tuple(map(float, m['comfort_temp_window']))
        self.eco = tuple(map(float, m['eco_temp_window']))
        self.morning = float(m['morning_start_hour'])
        self.evening = float(m['evening_start_hour'])
        normalizers = m.get('observation_normalization_metadata', {})
        temp_normalizer = normalizers.get('zone_air_temperature_sensor', {})
        self.temp_mean = float(temp_normalizer.get('sample_mean', 0.0))
        self.temp_std = float(temp_normalizer.get('sample_std', 1.0))
        if not np.isfinite(self.temp_std) or self.temp_std <= 0.0:
            self.temp_std = 1.0

        # Outdoor-air observations are NORMALIZED too. Without valid
        # normalization metadata, treat outdoor weather as unknown rather than
        # interpreting a z-score (e.g. -2.0) as a temperature in Kelvin.
        outdoor_normalizer = normalizers.get('outside_air_temperature_sensor', {})
        self.outdoor_mean = float(outdoor_normalizer.get('sample_mean', np.nan))
        self.outdoor_std = float(outdoor_normalizer.get('sample_std', np.nan))
        if (not np.isfinite(self.outdoor_mean) or
                not np.isfinite(self.outdoor_std) or
                self.outdoor_std <= 0.0):
            self.outdoor_mean = np.nan
            self.outdoor_std = np.nan

        def numeric_order(value):
            match = re.search(r'\d+', str(value))
            return (int(match.group()) if match else -1, str(value))

        self.ahus = []
        ahu_ids = sorted(m['ahu_zone_assignments'], key=numeric_order)[:n_ahu]
        for number, ahu_id in enumerate(ahu_ids):
            zones = []
            for room in m['ahu_zone_assignments'][ahu_id]:
                vav = str(room).replace('room_', 'VAV_')
                sensor = self.fields.get(vav + '_zone_air_temperature_sensor')
                if sensor is None:
                    continue
                actions = m['zone_action_indices'].get(vav, {})
                zones.append({
                    'name': vav,
                    'sensor': sensor,
                    'heat_setpoint': self.fields.get(
                        vav + '_zone_air_heating_temperature_setpoint_sensor'),
                    'cool_setpoint': self.fields.get(
                        vav + '_zone_air_cooling_temperature_setpoint_sensor'),
                    'damper_action': actions.get(
                        'supply_air_damper_percentage_command'),
                    'valve_action': actions.get(
                        'reheat_valve_percentage_command'),
                    'model': ZoneResponseModel(),
                })
            self.ahus.append({
                'zones': zones, 'run_action': number,
                'supply_action': self.ahu_supply_actions[number],
                'pressure_action': (
                    self.ahu_pressure_actions[number]
                    if number < len(self.ahu_pressure_actions) else None
                ),
                'recirc_action': (
                    self.ahu_recirc_actions[number]
                    if number < len(self.ahu_recirc_actions) else None
                ),
                'outside_sensor': self.fields.get(
                    'ahs_' + str(ahu_id) + '_outside_air_temperature_sensor'),
            })
        self.total_zones = max(1, sum(len(a['zones']) for a in self.ahus))
        self.number_ahus = max(1, len(self.ahus))

        # Simulator observation: normalized_count = (count-k)/(k+1).
        # Do not assume a fixed k: if metadata does not expose k, occupancy
        # remains uncertain and comfort is conservatively protected.
        scale = m.get('occupancy_normalization_constant')
        if scale is not None and np.isfinite(float(scale)) and float(scale) >= 0:
            scale = float(scale)
            self.empty_occupancy_signal = -scale / (scale + 1.0)
        else:
            self.empty_occupancy_signal = None
        self.reset()

    def reset(self):
        """Isolate episodes, including online model state and pending actions."""
        for ahu in self.ahus:
            for zone in ahu['zones']:
                zone['model'] = ZoneResponseModel()

    @staticmethod
    def _command(fraction):
        """Map physical actuator fraction [0, 1] to normalized [-1, 1]."""
        return float(np.clip(2.0 * fraction - 1.0, -1.0, 1.0))

    @staticmethod
    def _supply_command(kelvin):
        """Map 285..305 Kelvin supply-air setpoint into [-1, 1]."""
        return float(np.clip((kelvin - 285.0) / 10.0 - 1.0, -1.0, 1.0))

    def _fallback_band(self, obs):
        """Use weekday/hour only if current zone setpoint sensors are invalid."""
        hour = self._cyclic_clock(
            obs, self.fields, 'hod_sin_000', 'hod_cos_000', 24.0)
        day = self._cyclic_clock(
            obs, self.fields, 'dow_sin_000', 'dow_cos_000', 7.0)
        occupied = (hour is not None and day is not None and day < 5 and
                    self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zones(self, obs, ahu, fallback):
        """Denormalize zone temperatures, validate Kelvin setpoints, and learn.

        Observe the temperature response to the PREVIOUS issued action before
        selecting today's candidate. Invalid sensors clear pending transitions.
        """
        result = []
        outdoor_idx = ahu['outside_sensor']
        outdoor = np.nan
        if outdoor_idx is not None and np.isfinite(self.outdoor_std):
            normalized = float(obs[outdoor_idx])
            if np.isfinite(normalized):
                decoded = self.outdoor_mean + normalized * self.outdoor_std
                if 180.0 <= decoded <= 350.0:
                    outdoor = decoded
        for zone in ahu['zones']:
            temp = float(obs[zone['sensor']]) * self.temp_std + self.temp_mean
            if not np.isfinite(temp) or not 240.0 <= temp <= 340.0:
                zone['model'].clear_pending()
                continue
            li, ui = zone['heat_setpoint'], zone['cool_setpoint']
            low = float(obs[li]) if li is not None else np.nan
            high = float(obs[ui]) if ui is not None else np.nan
            if not (np.isfinite(low) and np.isfinite(high) and
                    245.0 <= low < high <= 335.0 and high-low <= 35.0):
                low, high = fallback
            if self.LEARN_DYNAMICS:
                zone['model'].observe(temp)
            result.append({'zone': zone, 'temp': temp, 'low': low,
                           'high': high, 'outdoor': outdoor})
        return result

    @staticmethod
    def _violation(temp, low, high):
        """Comfort deviation in K, with convex penalty beyond the 5K knee."""
        outside = max(low - temp, temp - high, 0.0)
        return outside / 5.0 if outside <= 5.0 else 1.0 + ((outside - 5.0) / 5.0)**2

    @staticmethod
    def _profiles(config, zone):
        """Small, commandable VAV grid for one AHU mode.

        With no reheat, search minimum ventilation, medium, and full airflow.
        With reheat, search minimum ventilation, full cooling, and full heat.
        Missing actuator mappings cannot be optimized as if controllable.
        """
        if not config.running:
            return ((0.0, 0.0),)

        can_heat = config.water_on and zone['valve_action'] is not None
        if zone['damper_action'] is None:
            return ((0.1, 0.0), (0.1, 1.0)) if can_heat else ((0.1, 0.0),)
        if can_heat:
            return ((0.1, 0.0), (1.0, 0.0), (1.0, 1.0))
        return ((0.1, 0.0), (0.55, 0.0), (1.0, 0.0))

    @staticmethod
    def _cyclic_clock(obs, fields, sine, cosine, period):
        """Decode observed cyclical clock without assuming a scenario start."""
        s, c = fields.get(sine), fields.get(cosine)
        if s is None or c is None or s >= len(obs) or c >= len(obs):
            return None
        a, b = float(obs[s]), float(obs[c])
        if not np.isfinite(a) or not np.isfinite(b):
            return None
        return (np.arctan2(a, b) % (2.0 * np.pi)) * period / (2.0 * np.pi)

    def _reward_context(self, obs):
        """Reward denominators/tariffs from public scorer; no future weather.

        The true reward uses per-zone occupancy, which cannot be observed.
        Approximate those weights as uniform, except for known-empty states.
        """
        hour = self._cyclic_clock(
            obs, self.fields, 'hod_sin_000', 'hod_cos_000', 24.0)
        day = self._cyclic_clock(
            obs, self.fields, 'dow_sin_000', 'dow_cos_000', 7.0)
        hour = 12 if hour is None else int(hour) % 24
        weekday = day is None or int(day) % 7 < 5
        if not weekday:
            electricity_usd_per_kwh = 0.16
        elif 6 <= hour < 12:
            electricity_usd_per_kwh = 0.18
        elif 12 <= hour < 19:
            electricity_usd_per_kwh = 0.20
        else:
            electricity_usd_per_kwh = 0.16

        carbon = self.ELECTRICITY_CO2_KG_PER_KWH[hour]
        occupancy_index = self.fields.get('num_occupants')
        activity = 1.0  # protect comfort when metadata is insufficient
        if (self.empty_occupancy_signal is not None
                and occupancy_index is not None
                and occupancy_index < len(obs)):
            raw = float(obs[occupancy_index])
            if (np.isfinite(raw) and
                    abs(raw - self.empty_occupancy_signal) <= 1e-5):
                activity = 0.0
        return electricity_usd_per_kwh, carbon, activity

    def _economic_energy_score(self, electric_kw, gas_kw, context):
        """Weighted normalized fuel cost and carbon; both nonnegative.

        Cost and carbon normalize by *separate* official-scale denominators,
        not by arbitrary kW weighting. Five-minute duration cancels between
        measured power and max-rate terms. Nominal scales are 80/205 kW per
        AHU in training configs; the hidden scenario may use other scales.
        """
        price, carbon, _ = context
        gas_price = self.GAS_USD_PER_KWH
        gas_carbon = self.GAS_KG_CO2_PER_KWH
        nominal_e = self.ELECTRICITY_NOMINAL_KW_PER_AHU * self.number_ahus
        nominal_g = self.GAS_NOMINAL_KW_PER_AHU * self.number_ahus
        cost_denominator = nominal_e * price + nominal_g * gas_price
        carbon_denominator = nominal_e * carbon + nominal_g * gas_carbon
        return (self.REWARD_COST *
                (electric_kw * price + gas_kw * gas_price) /
                max(cost_denominator, 1e-9) +
                self.REWARD_CARBON *
                (electric_kw * carbon + gas_kw * gas_carbon) /
                max(carbon_denominator, 1e-9))

    @staticmethod
    def _zone_power_kw(state, config, damper, valve, recirculation=0.70):
        """Conservative power proxy in kW; no fictitious energy observations.

        Max VAV airflow 2 m^3/s, air volumetric heat capacity 1.2 kJ/m^3/K.
        Fan electricity ~= pressure * airflow / (0.62*1000).
        Coil electricity uses COP 3.2. Gas reheat: estimated coil effectiveness
        0.35, boiler efficiency 0.85. These are uncalibrated physical priors.
        """
        if not config.running:
            return 0.0, 0.0
        temperature = state['temp']
        outdoor = state['outdoor']
        if not np.isfinite(outdoor):
            outdoor = temperature
        flow = 2.0 * max(0.10, damper)
        fan = config.fan_pa * flow / 620.0
        mixed = recirculation * temperature + (1.0 - recirculation) * outdoor
        delivered = min(config.supply_k, mixed)
        coil = 1.2 * flow * max(0.0, mixed - delivered) / 3.2
        gas = 0.0
        if config.water_on and valve > 0.0:
            delta = max(0.0, min(55.0, config.water_k - delivered))
            gas = 1.2 * flow * valve * delta * 0.35 / 0.85
        return fan + coil + (0.15 if gas > 0 else 0.0), gas

    def _best_vav(self, state, config, context, recirculation=0.70):
        """Optimize at most three profiles using a scalar linear model.

        Algebraically equivalent to V6's action-feature model at each fixed
        candidate: delta = base + damper*(cool_gain + valve*heat_gain).
        Reuse scalar thermal coefficients for every profile and checkpoint.
        Hard safety checks apply to the actual VAV profiles, not just the
        coarse AHU mode.
        """
        zone, temperature = state['zone'], state['temp']
        model = zone['model']
        weights = model.weights
        outdoor = state['outdoor']
        if not np.isfinite(outdoor):
            outdoor = temperature

        ambient = max(-3.0, min(3.0, (outdoor - temperature) / 20.0))
        base_rate = float(weights[0]) * ambient + float(weights[3])
        cool_gain = 0.0
        heat_gain = 0.0
        if config.running:
            delivered = min(
                config.supply_k,
                recirculation * temperature + (1.0 - recirculation) * outdoor)
            cooling_feature = max(
                0.0, min(2.0, (temperature - delivered) / 12.0))
            cool_gain = float(weights[1]) * cooling_feature
            if config.water_on:
                heating_feature = max(
                    0.0, min(2.0, (config.water_k - temperature) / 55.0))
                heat_gain = float(weights[2]) * heating_feature

        best = (float('inf'), 0.0, 0.0)
        for damper, valve in self._profiles(config, zone):
            # A heating-capable AHU is not enough: the actual VAV must reheat
            # an extremely cold zone. Likewise, strongly overheated zones
            # need meaningful cooling airflow without simultaneous reheat.
            # Skip only commands that can actually be controlled.
            if (state['low'] - temperature >= 4.0 and config.water_on
                    and zone['valve_action'] is not None and valve <= 0.0):
                continue
            if temperature - state['high'] >= 4.0:
                if valve > 0.0:
                    continue
                if (zone['damper_action'] is not None
                        and config.running and damper < 0.55):
                    continue

            rate = max(-0.50, min(
                0.50, base_rate + damper * (cool_gain + valve * heat_gain)))
            comfort = sum(self._violation(
                temperature + rate * steps, state['low'], state['high'])
                for steps in self.PLAN_STEPS) / len(self.PLAN_STEPS)
            if self.USE_ECONOMIC_OBJECTIVE:
                # Occupancy-weighted per-building mean discomfort (equal-zone
                # approximation), not a user-chosen comfort-energy exchange.
                score = (self.REWARD_COMFORT * context[2] * comfort /
                         self.total_zones)
                electric_kw, gas_kw = self._zone_power_kw(
                    state, config, damper, valve, recirculation)
                score += self._economic_energy_score(
                    electric_kw, gas_kw, context)
            else:
                # Exact V12 heuristic scoring is preserved for an A/B ablation.
                score = self.DISCOMFORT_WEIGHT * comfort
                if self.OPTIMIZE_ENERGY:
                    score += self.ENERGY_WEIGHT * (
                        (0.012 * damper if config.running else 0.0) +
                        (0.022 * damper * valve if config.water_on else 0.0))
            if score < best[0]:
                best = (score, damper, valve)
        return best

    @staticmethod
    def _plant_cost(config):
        """Estimate relative fan, cooling and boiler cost (not true kWh)."""
        if not config.running:
            return 0.0
        fan = 0.035 + 0.035 * (config.fan_pa / 2000.0)**1.5
        cooling = 0.025 * max(0.0, 299.0 - config.supply_k) / 14.0
        heating = (0.025 * max(0.0, config.water_k - 310.0) / 40.0
                   if config.water_on else 0.0)
        return fan + cooling + heating

    @staticmethod
    def _candidate_allowed(config, worst_hot, worst_cold):
        """Keep a corrective mode available for every severe demand.

        Off is forbidden at a 2 K violation. Severe hot/cold demands require
        a mode that can correct that direction. Mixed stays feasible when
        zones demand both cooling and heating.
        """
        if config.mode == 'off' and max(worst_hot, worst_cold) >= 2.0:
            return False
        # Severe excursions require a corrective plant mode. Mixed mode
        # remains feasible if hot and cold zones coexist.
        if worst_hot >= 4.0 and config.running and config.supply_k >= 300.0:
            return False
        if worst_cold >= 4.0 and not config.water_on:
            return False
        return True

    def _select_operating_point(self, zones, context, has_recirculation=True):
        """Select an AHU mode plus per-zone VAV profiles by short search.

        Nonnegative reward terms allow early pruning of a candidate.
        This finds the best mode in the bounded grid, not the global
        optimum of the HVAC plant.
        """
        worst_hot = max((z['temp'] - z['high'] for z in zones), default=0.0)
        worst_cold = max((z['low'] - z['temp'] for z in zones), default=0.0)
        best_cost, best_config, best_vavs = float('inf'), None, None
        zone_count = len(zones)

        for config in self.OPERATING_POINTS:
            if not self._candidate_allowed(config, worst_hot, worst_cold):
                continue
            if self.USE_ECONOMIC_OBJECTIVE:
                # Fixed fan/control overhead; zone-dependent fan/coil energy
                # is added by _best_vav, avoiding duplicate consumption.
                cost = self._economic_energy_score(
                    1.2 if config.running else 0.0, 0.0, context)
            else:
                cost = (self.ENERGY_WEIGHT * self._plant_cost(config)
                        if self.OPTIMIZE_ENERGY else 0.0)
            recirculation = self._planned_recirculation(
                config, zones, has_recirculation)
            choices = []
            for state in zones:
                choice = self._best_vav(
                    state, config, context, recirculation)
                choices.append(choice)
                # The economic objective already normalizes by the whole
                # building's zone count and total rated power. Dividing again
                # by this AHU's zone count would erroneously discount comfort
                # and fuel on large AHUs. The V12 ablation remains averaged.
                cost += (choice[0] if self.USE_ECONOMIC_OBJECTIVE
                         else choice[0] / zone_count)
                if cost >= best_cost:
                    break
            if len(choices) == zone_count and cost < best_cost:
                best_cost, best_config, best_vavs = cost, config, choices
        return best_config, best_vavs

    @staticmethod
    def _planned_recirculation(config, zones, has_actuator=True):
        """Plan physical return-air fraction, matching forecast and command.

        The V12 economizer chose its mix after optimization, so predicted
        chiller energy could disagree with the action that was actually sent.
        """
        if (not config.running or not has_actuator or
                config.supply_k >= 300.0 or not zones):
            return 0.70
        outdoor = zones[0]['outdoor']
        zone_mean = sum(s['temp'] for s in zones) / len(zones)
        if (not np.isfinite(outdoor) or outdoor >= zone_mean or
                zone_mean - outdoor <= config.fan_pa * 0.00322):
            return 0.70
        outside_fraction = np.clip(
            (zone_mean - config.supply_k) / max(1e-6, zone_mean - outdoor),
            0.30, 1.0)
        return float(np.clip(1.0 - outside_fraction, 0.0, 0.70))

    def _initial_actions(self):
        """Allocate safe idle commands, with correct array sizes and types.

        A normalized continuous zero is the MIDPOINT of a physical range,
        not OFF. +1 requests warmest AHU supply and maximum recirculation.
        """
        continuous = np.full(self.continuous_count, -1.0, dtype=np.float32)
        discrete = np.zeros(self.discrete_count, dtype=np.int32)
        for action_index in self.ahu_supply_actions + self.ahu_recirc_actions:
            continuous[action_index] = 1.0
        return continuous, discrete

    def _apply_ahu_action(self, ahu, config, recirculation, continuous, discrete):
        """Command one AHU's run flag, cooling supply, fan and economizer.

        AHUs cannot heat: thermal reheat occurs downstream in the VAV boxes.
        """
        if config.running:
            discrete[ahu['run_action']] = 1
            continuous[ahu['supply_action']] = self._supply_command(
                config.supply_k)
            if ahu['pressure_action'] is not None:
                continuous[ahu['pressure_action']] = self._command(
                    config.fan_pa / 2000.0)
            if ahu['recirc_action'] is not None:
                continuous[ahu['recirc_action']] = self._command(recirculation)

    def _apply_vav_actions(self, states, config, choices, recirculation,
                           continuous, heat_demands, pending):
        """Issue achievable zone commands and collect model update inputs.

        We defer saving the model's features until the shared boiler command
        is known: different AHUs can request different water temperatures.
        """
        for state, (_, damper, valve) in zip(states, choices):
            zone = state['zone']
            damper_index = zone['damper_action']
            valve_index = zone['valve_action']
            if valve_index is None:
                valve = 0.0
            if damper_index is None:
                damper = 0.10 if config.running else 0.0

            if damper_index is not None:
                continuous[damper_index] = self._command(damper)
            if valve_index is not None:
                continuous[valve_index] = self._command(valve)
            if config.running and config.water_on and valve > 0.0:
                heat_demands.append((config.water_k, config.fan_pa,
                                     damper * valve))

            pending.append((state, config, damper, valve, recirculation))

    def _apply_shared_hot_water(self, heat_demands, continuous, discrete):
        """Combine all AHUs' valid reheat requests into one water command.

        Returns the common commanded water temperature, or None if unused.
        """
        if not heat_demands:
            return None

        discrete[self.water_run_action] = 1
        hot_water = max(demand[0] for demand in heat_demands)
        loading = float(np.mean([demand[2] for demand in heat_demands]))
        if self.water_temp_action is not None:
            continuous[self.water_temp_action] = self._command(
                (hot_water - 310.0) / 40.0)
        if self.water_pressure_action is not None:
            continuous[self.water_pressure_action] = self._command(
                max(0.35, loading))
        return hot_water

    @staticmethod
    def _remember_issued_actions(pending, common_water_k):
        """Store jointly issued setpoints for the next RLS update.

        These are commands, not necessarily the measured actuator positions:
        the simulator can impose floors and thermal response takes time.
        """
        for state, config, damper, valve, recirculation in pending:
            heat_enabled = (common_water_k is not None and config.running
                            and config.water_on and valve > 0.0)
            water_k = common_water_k if heat_enabled else config.water_k
            features = state['zone']['model'].features(
                state['temp'], state['outdoor'], float(config.running),
                config.supply_k, damper, valve,
                float(heat_enabled), water_k, recirculation)
            state['zone']['model'].remember(state['temp'], features)

    def get_action(self, obs):
        """One step: learn previous outcome -> plan -> issue -> remember.

        (1) Decode sensors and learn from the preceding issued commands.
        (2) Search six AHU modes and at most three VAV profiles per zone.
        (3) Issue AHU/VAV commands and coordinate their shared hot-water loop.
        (4) Save the joint issued action for the next model observation.
        (5) Return normalized, typed hybrid action arrays.
        """
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._initial_actions()
        fallback = self._fallback_band(obs)
        context = self._reward_context(obs)
        heat_demands = []
        pending = []

        for ahu in self.ahus:
            states = self._read_zones(obs, ahu, fallback)
            if not states:
                continue
            has_recirculation = ahu['recirc_action'] is not None
            config, choices = self._select_operating_point(
                states, context, has_recirculation)
            recirculation = self._planned_recirculation(
                config, states, has_recirculation)
            self._apply_ahu_action(
                ahu, config, recirculation, continuous, discrete)
            self._apply_vav_actions(
                states, config, choices, recirculation,
                continuous, heat_demands, pending)

        common_water_k = self._apply_shared_hot_water(
            heat_demands, continuous, discrete)
        self._remember_issued_actions(pending, common_water_k)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {'continuous_action': continuous, 'discrete_action': discrete}


In [ ]:
# Competition gateway: keep this as the final Kaggle notebook cell.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
